In [30]:
import pandas as pd
import numpy as np
import sys
sys.path.insert(0, '..')
df = pd.read_csv('../data/raw/reddit_depression_dataset.csv')

/var/folders/zp/4dbbl8053pv2zgxvxs73yzbh0000gn/T/ipykernel_82481/3201626537.py:5: DtypeWarning: Columns (0: Unnamed: 0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/raw/reddit_depression_dataset.csv')


In [3]:
df.head()

,Unnamed: 0,subreddit,title,body,upvotes,created_utc,num_comments,label
0,47951,DeepThoughts,Deep thoughts underdog,"Only when we start considering ourselves, the ...",4.0,1.405309e+09,NaN,0.0
1,47952,DeepThoughts,"I like this sub, there's only two posts yet I ...",Anyway: Human Morality is a joke so long as th...,4.0,1.410568e+09,1.0,0.0
2,47957,DeepThoughts,Rebirth!,Hello. \nI am the new guy in charge here (Besi...,6.0,1.416458e+09,1.0,0.0
3,47959,DeepThoughts,"""I want to be like water. I want to slip throu...",NaN,25.0,1.416512e+09,2.0,0.0
4,47960,DeepThoughts,Who am I?,You could take any one cell in my body and kil...,5.0,1.416516e+09,4.0,0.0


In [4]:
df.subreddit.value_counts()

subreddit
teenagers       1956521
depression       290058
SuicideWatch     190364
happy             24609
DeepThoughts       9163
4                    10
6                     5
5                     5
30                    2
16                    2
7                     2
10                    2
9                     2
8                     2
11                    2
15                    1
12                    1
61                    1
107                   1
1402326041            1
31                    1
101                   1
27                    1
47                    1
Name: count, dtype: int64

In [6]:
df_emotion = pd.read_csv('../data/raw/emotion_train.csv')  

In [8]:
df_emotion.shape

(16000, 2)

In [9]:
df_emotion_unsplit = pd.read_csv('../data/raw/emotion_unsplit.csv')

In [10]:
df_emotion_unsplit.head(5)

,text,label
0,i feel awful about it too because it s my job ...,0
1,im alone i feel awful,0
2,ive probably mentioned this before but i reall...,1
3,i was feeling a little low few days back,0
4,i beleive that i am much more sensitive to oth...,2


In [15]:
df_emotion_unsplit.label.value_counts()

df_emotion_unsplit[df_emotion_unsplit.label == 0].head(5)

,text,label
0,i feel awful about it too because it s my job ...,0
1,im alone i feel awful,0
3,i was feeling a little low few days back,0
11,i also feel disillusioned that someone who cla...,0
16,i wish you knew every word i write i write for...,0


In [17]:
df_emotion_unsplit.label.value_counts()

label
1    141067
0    121187
3     57317
4     47712
2     34554
5     14972
Name: count, dtype: int64

In [ ]:
def balance_classes(df, label_col, random_state=42):
    class_counts = df[label_col].value_counts()
    min_count = class_counts.min()
    balanced_df = pd.DataFrame()
    
    for label in class_counts.index:
        label_df = df[df[label_col] == label]
        balanced_label_df = label_df.sample(min_count, random_state=random_state)
        balanced_df = pd.concat([balanced_df, balanced_label_df])
    
    return balanced_df.sample(frac=1, random_state=random_state).reset_index(drop=True)

In [36]:
df_emotion_cleaned = df_emotion_unsplit[df_emotion_unsplit.label != 4]
df_emotion_cleaned.loc[df_emotion_cleaned.label != 0, 'label'] = 1

df_emotion_balanced = balance_classes(df_emotion_cleaned, 'label')

df_emotion_balanced.label.value_counts()


label
1    121187
0    121187
Name: count, dtype: int64

In [ ]:
semeval_df = pd.read_csv('../data/raw/semeval.csv', delimiter='\t')
semeval_df.head(5)



,ID,Tweet,anger,anticipation,disgust,fear,joy,love,optimism,pessimism,sadness,surprise,trust
0,2018-En-00866,"@RanaAyyub @rajnathsingh Oh, hidden revenge an...",1,0,1,0,0,0,0,0,0,0,0
1,2018-En-02590,I'm doing all this to make sure you smiling do...,0,0,0,0,1,1,1,0,0,0,0
2,2018-En-03361,if not then #teamchristine bc all tana has don...,1,0,1,0,0,0,0,0,0,0,0
3,2018-En-03230,It is a #great start for #beginners to jump in...,0,0,0,0,1,0,1,0,0,0,0
4,2018-En-01143,My best friends driving for the first time wit...,0,0,0,1,0,0,0,0,0,0,0


In [38]:
semeval_df[semeval_df['sadness'] == 1].head(10)
semeval_df['label'] = np.where(((semeval_df['sadness'] == 1) | (semeval_df['pessimism'] == 1)) 
                               & (semeval_df['joy'] != 1) & (semeval_df['love'] != 1) 
                               & (semeval_df['optimism'] != 1) & (semeval_df['trust'] != 1), 1, 0)
semeval_df.label.value_counts()

semeval_df_balanced = balance_classes(semeval_df, 'label')[['Tweet', 'label']]

In [31]:
import src.data_cleaning.data as data_utils

df_cleaned = data_utils.clean_data(df)



In [32]:
df_cleaned.head()

df_cleaned.label.value_counts()

label
0.0    1990260
1.0     480409
Name: count, dtype: int64

def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    # Remove duplicates
    df = df.drop_duplicates()
    #Remove na_values for those who haven't not a lot of na
    df = df.dropna(subset=["Unnamed: 0","subreddit","title",
                       "upvotes", "created_utc", "label"], axis=0)
    #Inversion de certaines valeurs de Unnamed: 0 et title
    unnamed_is_numeric = pd.to_numeric(df["Unnamed: 0"], errors="coerce").notna()
    title_is_numeric = pd.to_numeric(df["title"], errors="coerce").notna()
    mask = (~unnamed_is_numeric) & title_is_numeric

    # 3) Inversion des valeurs entre les 2 colonnes sur ces lignes
    df.loc[mask, ["Unnamed: 0", "title"]] = (
        df.loc[mask, ["title", "Unnamed: 0"]].to_numpy())
    #On drop les colonnes qui ne sont pas utiles
    #On va fusionner les colonnes de texte body et title 
    sub_df = df[["title", "body"]]
    text = sub_df["title"] + " " + sub_df["body"]
    df["title"] = text
    df.drop(columns=["Unnamed: 0", "body", "subreddit